# 03 - Capitulation probes (Q3)

Train logistic-regression probes on early-layer (1-8) and middle-layer
(9-16) activations, predicting whether the victim will capitulate by the
final turn. Compares against majority-class and permutation baselines.
If early-layer probes are significantly above chance, the capitulation
decision is encoded BEFORE the model expresses any reasoning -- evidence
that expressed CoT is post-hoc with respect to capitulation.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils.io import load_jsonl, load_yaml
from src.analysis.probes import assemble_probe_data, probe_per_layer

In [ ]:
RUNS_DIR = pathlib.Path('../runs')
cfg = load_yaml('../config/default.yaml')
results_all = []
for ef in RUNS_DIR.rglob('exchanges.jsonl'):
    exchanges = load_jsonl(ef)
    if not exchanges:
        continue
    layers_data = assemble_probe_data(exchanges, ef, use_turn='first')
    res = probe_per_layer(layers_data, layers=range(0, 32), permutation_n=200)
    res['victim'] = exchanges[0]['victim_name']
    results_all.append(res)
res = pd.concat(results_all, ignore_index=True)
res.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for victim, sub in res.groupby('victim'):
    ax.plot(sub['layer'], sub['accuracy'], marker='o', label=victim)
ax.axhline(res['majority'].iloc[0], ls='--', color='gray', label='majority baseline')
ax.set_xlabel('layer'); ax.set_ylabel('probe accuracy (5-fold CV)')
ax.set_title('Capitulation probe accuracy at turn-1 activations')
ax.legend(); plt.tight_layout()

In [ ]:
# Significant layers (Bonferroni-adjusted threshold over 32 layers)
alpha = 0.05 / max(1, len(res))
res_sig = res[res['permutation_p'] < alpha].sort_values('accuracy', ascending=False)
res_sig.head(10)